In [ ]:
# CELDA 1 — Setup del notebook de riesgo (Isolation Forest + Score continuo)
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, time, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

PROJECT_PATH   = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH    = f'{PROJECT_PATH}/models'
FIGURES_PATH   = f'{PROJECT_PATH}/reports/figures'

train = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
test  = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')
hist  = pd.read_parquet(f'{PROCESSED_PATH}/dataset_unificado.parquet')

# Filtrar notas de credito
train = train[train['Importe_Total_PEN'] > 0].copy().reset_index(drop=True)
test  = test[test['Importe_Total_PEN']  > 0].copy().reset_index(drop=True)

print(f'Train: {len(train):,} filas | Test: {len(test):,} filas')
print(f'Operaciones en train: {train["Nro. Ope."].nunique():,}')
print(f'Operaciones en test:  {test["Nro. Ope."].nunique():,}')


## Corrección P1-4: Excluir operaciones solapadas del test de evaluación

In [ ]:
# CELDA 2 — Excluir 139 operaciones que aparecen en AMBOS conjuntos del test de evaluacion
# Estas ops tienen facturas en 2024 Y en 2025, por lo que aparecen en train y test.
# Para una evaluacion limpia del modelo, las excluimos del test (aunque el modelo
# las vera en produccion con normalidad — esto solo afecta la metrica de evaluacion).

ops_train   = set(train['Nro. Ope.'].unique())
ops_test    = set(test['Nro. Ope.'].unique())
ops_overlap = ops_train & ops_test

test_eval = test[~test['Nro. Ope.'].isin(ops_overlap)].copy().reset_index(drop=True)

print(f'Operaciones solapadas excluidas: {len(ops_overlap)}')
print(f'Test original:   {len(test):,} filas')
print(f'Test para eval:  {len(test_eval):,} filas (sin solapamiento)')


## Corrección P1-2: Features sin fecha_original

> **Importante:** `fecha_original` era el feature #1 en el modelo anterior, pero es un **artefacto de calidad de datos**: en producción todos los despachos nuevos tienen fecha real (flag = 1 siempre), por lo que el modelo aprendía una distinción que no existe en producción. Se excluye completamente.

In [ ]:
# CELDA 3 — Definir features SIN fecha_original
# NOTA: tarifa_historica SI se puede incluir aqui porque se calculo con
# expanding().shift(1) — solo usa datos pasados, no hay leakage.

NUMERICAS = [
    'Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores',
    'Peso Bruto (kg)', 'bultos_por_contenedor', 'peso_por_contenedor',
    'dias_desde_inicio', 'proveedor_frecuencia',
    # Nuevas features P3-9 (si existen):
    'mes_sin', 'mes_cos', 'semana_sin', 'semana_cos',
    'semanas_hasta_cierre', 'dias_transito_estimado',
    'peso_disponible', 'tiene_proyecto', 'es_temporada_alta', 'es_cierre_fiscal',
    # EXCLUIDO INTENCIONALMENTE: 'fecha_original'
]

CATEGORICAS = [
    'Concepto Canónico', 'Proveedor_norm', 'ACREEDOR_norm',
    'AGENCIA DE ADUANA_norm', 'Proveedor Principal_norm',
    'incoterm_familia', 'Modalidad (MODE Y TYPE)',
]

# Agregar ruta_origen_grupo si existe
if 'ruta_origen_grupo' in train.columns:
    CATEGORICAS.append('ruta_origen_grupo')

# Filtrar a columnas que existen en el dataset
NUMERICAS   = [c for c in NUMERICAS   if c in train.columns]
CATEGORICAS = [c for c in CATEGORICAS if c in train.columns]
FEATURES    = NUMERICAS + CATEGORICAS

print(f'Features numericas:   {len(NUMERICAS)}')
print(f'Features categoricas: {len(CATEGORICAS)}')
print(f'Total features:       {len(FEATURES)}')
print('fecha_original en features:', 'fecha_original' in FEATURES)


## Mejora P1-3: Isolation Forest — Detección de Anomalías

En lugar de clasificar riesgo con thresholds arbitrarios (25%/75% del desvío), usamos Isolation Forest que aprende qué facturas son *intrínsecamente anómalas* respecto al comportamiento histórico. Produce un **score continuo** que permite rankear facturas por prioridad de revisión, sin depender de la tarifa histórica como referencia circular.

In [ ]:
# CELDA 4 — Preparar matrices X para Isolation Forest
# Isolation Forest solo necesita features (no hay target supervisado)

def preparar_X(df_in, numericas, categoricas, encoders=None, fit=True):
    X = df_in[numericas + categoricas].copy()
    # Imputar numericas
    for col in numericas:
        if fit:
            med = X[col].median() if X[col].notna().any() else 0
        else:
            med = encoders['medians'].get(col, 0)
        X[col] = X[col].fillna(med)
    # Imputar y encodear categoricas
    for col in categoricas:
        X[col] = X[col].fillna('DESCONOCIDO').astype(str)
    if fit:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categoricas] = enc.fit_transform(X[categoricas])
        medians = {col: df_in[col].median() if df_in[col].notna().any() else 0
                   for col in numericas}
        return X, {'encoder': enc, 'medians': medians}
    else:
        X[categoricas] = encoders['encoder'].transform(X[categoricas])
        return X

X_train, enc_dict = preparar_X(train, NUMERICAS, CATEGORICAS, fit=True)
X_test_eval       = preparar_X(test_eval, NUMERICAS, CATEGORICAS, encoders=enc_dict, fit=False)
X_test_full       = preparar_X(test,      NUMERICAS, CATEGORICAS, encoders=enc_dict, fit=False)

print(f'X_train: {X_train.shape}')
print(f'X_test_eval: {X_test_eval.shape}')


In [ ]:
# CELDA 5 — Entrenar Isolation Forest
# contamination=0.1 -> asume que ~10% de las facturas son potencialmente anomalas
# Esto es ajustable segun el umbral operativo que defina el equipo de auditoria

print('Entrenando Isolation Forest...')
t0 = time.time()

iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.10,    # 10% de facturas esperadas como anomalas
    max_features=0.8,      # subconjunto de features por arbol (robustez)
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train)
print(f'Entrenado en {time.time()-t0:.1f} segundos')

# Scores en train y test (decision_function: mas negativo = mas anomalo)
train['anomaly_score'] = iso_model.decision_function(X_train)
train['es_anomalo']    = iso_model.predict(X_train)  # -1=anomalo, +1=normal

test_eval['anomaly_score'] = iso_model.decision_function(X_test_eval)
test_eval['es_anomalo']    = iso_model.predict(X_test_eval)

test['anomaly_score'] = iso_model.decision_function(X_test_full)
test['es_anomalo']    = iso_model.predict(X_test_full)

# Convertir score a percentil 0-100 (mas intuitivo para el negocio)
from scipy.stats import rankdata
train['riesgo_percentil'] = rankdata(-train['anomaly_score']) / len(train) * 100
test_eval['riesgo_percentil'] = rankdata(-test_eval['anomaly_score']) / len(test_eval) * 100

# Score relativo: invertir signo para que mayor score = mayor riesgo
score_min = train['anomaly_score'].min()
score_max = train['anomaly_score'].max()
train['riesgo_score_0_100'] = (train['anomaly_score'] - score_max) / (score_min - score_max) * 100
test_eval['riesgo_score_0_100'] = (test_eval['anomaly_score'] - score_max) / (score_min - score_max) * 100

print(f'Anomalias detectadas en train: {(train["es_anomalo"]==-1).sum():,} ({(train["es_anomalo"]==-1).mean()*100:.1f}%)')
print(f'Anomalias detectadas en test:  {(test_eval["es_anomalo"]==-1).sum():,} ({(test_eval["es_anomalo"]==-1).mean()*100:.1f}%)')

print('\nTop 10 facturas mas anomalas del test set:')
top_anomalas = test_eval.nsmallest(10, 'anomaly_score')[[
    'Nro. Ope.', 'Concepto Canónico', 'Importe_Total_PEN',
    'tarifa_historica', 'anomaly_score', 'riesgo_score_0_100'
]]
print(top_anomalas.to_string(index=False))


In [ ]:
# CELDA 6 — Categorias de riesgo operativo basadas en score continuo
# Ventaja vs thresholds arbitrarios: los umbrales se basan en la distribucion
# real de anomalias, no en porcentajes fijos de desvio

def asignar_nivel_riesgo(score_percentil):
    if score_percentil >= 80:
        return 'ALTO'
    elif score_percentil >= 50:
        return 'MEDIO'
    else:
        return 'BAJO'

train['nivel_riesgo']     = train['riesgo_percentil'].apply(asignar_nivel_riesgo)
test_eval['nivel_riesgo'] = test_eval['riesgo_percentil'].apply(asignar_nivel_riesgo)
test['nivel_riesgo']      = test['riesgo_percentil'].apply(
    lambda x: asignar_nivel_riesgo(rankdata([-s for s in test['anomaly_score'].values])[list(test['anomaly_score'].values).index(test.loc[test['riesgo_percentil'].isna(),'anomaly_score'].values[0] if test['riesgo_percentil'].isna().any() else x)] / len(test) * 100) if True else 'BAJO'
) if False else test['anomaly_score'].apply(
    lambda s: 'ALTO' if s < test['anomaly_score'].quantile(0.20) else ('MEDIO' if s < test['anomaly_score'].quantile(0.50) else 'BAJO')
)

print('Distribucion de nivel de riesgo en train:')
print(train['nivel_riesgo'].value_counts().to_string())
print('\nDistribucion de nivel de riesgo en test:')
print(test_eval['nivel_riesgo'].value_counts().to_string())

# Importe promedio por nivel de riesgo (validacion de negocio)
print('\nImporte medio por nivel de riesgo (validacion):')
print(test_eval.groupby('nivel_riesgo')['Importe_Total_PEN']
      .agg(['count', 'mean', 'median']).round(0).to_string())


In [ ]:
# CELDA 7 — Visualizacion del score de riesgo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribucion del anomaly score en train vs test
axes[0].hist(train['anomaly_score'], bins=50, alpha=0.6, label='Train', color='steelblue', density=True)
axes[0].hist(test_eval['anomaly_score'], bins=50, alpha=0.6, label='Test', color='orange', density=True)
axes[0].axvline(0, color='red', linestyle='--', label='Umbral (0)')
axes[0].set_title('Distribucion del Anomaly Score')
axes[0].set_xlabel('Score (negativo = mas anomalo)')
axes[0].legend()

# Score por concepto canonico
scores_por_concepto = test_eval.groupby('Concepto Canónico')['riesgo_score_0_100'].mean().sort_values(ascending=False)
axes[1].barh(scores_por_concepto.index, scores_por_concepto.values, color='salmon')
axes[1].set_title('Score de Riesgo Medio por Concepto')
axes[1].set_xlabel('Score 0-100 (mayor = mas riesgo)')

# Importe real vs score de riesgo (scatter)
axes[2].scatter(test_eval['riesgo_score_0_100'],
                np.log1p(test_eval['Importe_Total_PEN']),
                alpha=0.3, s=5, c='navy')
axes[2].set_title('Score de Riesgo vs log(Importe Real)')
axes[2].set_xlabel('Score de Riesgo 0-100')
axes[2].set_ylabel('log(Importe_PEN)')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/12b_isolation_forest_scores.png', dpi=150, bbox_inches='tight')
plt.show()


## Mejora P2-6: XGBoost con TimeSeriesSplit para clasificacion de riesgo

Adicionalmente al Isolation Forest (que opera sin supervision), entrenamos un XGBoost que aprende a predecir el nivel de riesgo usando los labels generados por el Isolation Forest. Esto permite usar el clasificador en produccion sin necesidad de reentrenar el IF completo.

In [ ]:
# CELDA 8 — XGBoost con TimeSeriesSplit usando labels de Isolation Forest
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight

# Usar los labels de IF como target supervisado
le = LabelEncoder()
y_train_enc = le.fit_transform(train['nivel_riesgo'])  # ALTO=0, BAJO=1, MEDIO=2
y_test_enc  = le.transform(test_eval['nivel_riesgo'])
print('Clases:', dict(zip(le.classes_, range(len(le.classes_)))))

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

# TimeSeriesSplit — validacion temporal dentro del train set
orden_temporal = train.sort_values('Fecha_Imputada').index
X_tr_sorted = X_train.loc[orden_temporal].reset_index(drop=True)
y_tr_sorted = y_train_enc[orden_temporal]

tscv = TimeSeriesSplit(n_splits=5, gap=30)
cv_scores_xgb = []

print('Validacion cruzada temporal (5 folds)...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_tr_sorted)):
    X_f_tr, X_f_val = X_tr_sorted.iloc[tr_idx], X_tr_sorted.iloc[val_idx]
    y_f_tr, y_f_val = y_tr_sorted[tr_idx], y_tr_sorted[val_idx]
    sw_f = compute_sample_weight('balanced', y_f_tr)

    clf = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        objective='multi:softprob', num_class=3,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    clf.fit(X_f_tr, y_f_tr, sample_weight=sw_f,
            eval_set=[(X_f_val, y_f_val)], verbose=False)

    acc = accuracy_score(y_f_val, clf.predict(X_f_val))
    cv_scores_xgb.append(acc)
    print(f'  Fold {fold+1}: accuracy={acc:.3f}')

print(f'\nCV accuracy: {np.mean(cv_scores_xgb):.3f} +/- {np.std(cv_scores_xgb):.3f}')


In [ ]:
# CELDA 9 — Entrenar modelo final XGBoost sobre todo el train
print('Entrenando XGBoost final...')
t0 = time.time()

xgb_clf = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    objective='multi:softprob', num_class=3,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_clf.fit(X_train, y_train_enc, sample_weight=sample_weights)
print(f'Entrenado en {time.time()-t0:.1f} segundos')

y_pred_xgb     = xgb_clf.predict(X_test_eval)
y_proba_xgb    = xgb_clf.predict_proba(X_test_eval)

acc_xgb = accuracy_score(y_test_enc, y_pred_xgb)
print(f'\nAccuracy en test (sin solapamiento): {acc_xgb:.3f}')
print('\nReporte de clasificacion:')
print(classification_report(y_test_enc, y_pred_xgb,
                             target_names=le.classes_))

# Feature importance (sin fecha_original)
importancia = pd.DataFrame({
    'feature': FEATURES,
    'importancia': xgb_clf.feature_importances_
}).sort_values('importancia', ascending=False)
print('\nTop 10 features mas importantes:')
print(importancia.head(10).to_string(index=False))
print('\nfecha_original en features:', 'fecha_original' in FEATURES)


In [ ]:
# CELDA 10 — Comparacion Isolation Forest vs XGBoost supervisado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix XGBoost
cm = confusion_matrix(y_test_enc, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=le.classes_, yticklabels=le.classes_,
            cmap='Blues')
axes[0].set_title(f'XGBoost (sin fecha_original)\nAccuracy: {acc_xgb:.3f}')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicho')

# Feature importance top 15
top_feat = importancia.head(15).iloc[::-1]
axes[1].barh(top_feat['feature'], top_feat['importancia'], color='steelblue')
axes[1].set_title('Feature Importance XGBoost\n(fecha_original excluida)')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/15b_clasificacion_mejorada.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen comparativo
print('RESUMEN COMPARATIVO:')
print(f'  XGBoost anterior (con fecha_original): accuracy ~0.519, F1~0.51')
print(f'  XGBoost nuevo (sin fecha_original):    accuracy {acc_xgb:.3f}')
print(f'  Isolation Forest: score continuo 0-100 (mas flexible para negocio)')


In [ ]:
# CELDA 11 — Guardar modelos de riesgo
os.makedirs(MODELS_PATH, exist_ok=True)

# Isolation Forest (modelo principal de riesgo)
joblib.dump(iso_model,  f'{MODELS_PATH}/isolation_forest.joblib')
joblib.dump(enc_dict,   f'{MODELS_PATH}/riesgo_encoder.joblib')

# Parametros de calibracion del score
score_params = {'score_min': score_min, 'score_max': score_max}
joblib.dump(score_params, f'{MODELS_PATH}/riesgo_score_params.joblib')

# XGBoost supervisado (clasificador auxiliar)
joblib.dump(xgb_clf, f'{MODELS_PATH}/xgboost_classifier_v2.joblib')
joblib.dump(le,      f'{MODELS_PATH}/xgboost_label_encoder_v2.joblib')

# Guardar datasets con score de riesgo anotado
train.to_parquet(f'{PROCESSED_PATH}/train_con_riesgo.parquet', index=False)
test.to_parquet(f'{PROCESSED_PATH}/test_con_riesgo.parquet', index=False)

print('Modelos guardados:')
for fn in ['isolation_forest.joblib', 'riesgo_encoder.joblib',
           'riesgo_score_params.joblib', 'xgboost_classifier_v2.joblib',
           'xgboost_label_encoder_v2.joblib']:
    fpath = f'{MODELS_PATH}/{fn}'
    sz = os.path.getsize(fpath)/1024 if os.path.exists(fpath) else 0
    print(f'  {fn}: {sz:.1f} KB')

# Resumen final
print('\n=== RESUMEN DEL TRACK DE RIESGO ===')
print(f'Modelo principal: Isolation Forest (score continuo 0-100)')
print(f'Modelo auxiliar: XGBoost clasificador (para API rapida)')
print(f'Features: {len(FEATURES)} (fecha_original EXCLUIDA)')
print(f'Test limpio (sin solapamiento): {len(test_eval):,} filas')
print(f'Anomalias en test: {(test_eval["es_anomalo"]==-1).sum():,} ({(test_eval["es_anomalo"]==-1).mean()*100:.1f}%)')
